# Task 1: fine-tuning Qwen3-0.6B on MetaMathQA

Everything that has to survive a disconnect (checkpoints, exported models, results) is written to Google Drive.
Training resumes automatically: just re-run the same training cell after a disconnect.

Suggested runtime: **L4** for training and evaluation, T4 is fine for the smoke test of the LoRA runs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/atml_pa3/task1'
RUNS, EVAL, RES = f'{ROOT}/runs', f'{ROOT}/eval_models', f'{ROOT}/results'
for d in (RUNS, EVAL, RES):
    os.makedirs(d, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

Clone the repo. If it is still private, add a GitHub token as a Colab secret named `GH_TOKEN` and use the second form.

In [ ]:
REPO = 'github.com/umairs5/atml-pa3.git'
# from google.colab import userdata; REPO = f"{userdata.get('GH_TOKEN')}@{REPO}"
if not os.path.exists('/content/pa3'):
    !git clone -q https://{REPO} /content/pa3
else:
    !cd /content/pa3 && git pull -q
%cd /content/pa3
!pip install -q -r requirements.txt

## Training data

Built once (about a minute on CPU) and cached on Drive. `data_info.json` should report 40,000 kept examples and 9,908,423 total tokens.

In [ ]:
DATA = f'{ROOT}/data/metamath_40k.npz'
if not os.path.exists(DATA):
    !python -m task1.prepare_data --out {ROOT}/data
!mkdir -p /content/data && cp {DATA} /content/data/
!cat {ROOT}/data/data_info.json

## Smoke test

A few real updates to check memory and seconds per update before committing to a full run. Writes to local disk only.

In [ ]:
!python -m task1.train --config task1/configs/full_ft_l2sp.yaml --out_root /content/smoke --data /content/data/metamath_40k.npz --max_updates 3
!python -m task1.train --config task1/configs/oplora_r16.yaml --out_root /content/smoke --data /content/data/metamath_40k.npz --max_updates 3
!rm -rf /content/smoke

## Training runs

One run per cell. Re-running a cell continues from the latest checkpoint.

In [ ]:
!python -m task1.train --config task1/configs/full_ft.yaml --out_root {RUNS} --data /content/data/metamath_40k.npz

In [ ]:
!python -m task1.train --config task1/configs/full_ft_l2sp.yaml --out_root {RUNS} --data /content/data/metamath_40k.npz

In [ ]:
!python -m task1.train --config task1/configs/lora_r16.yaml --out_root {RUNS} --data /content/data/metamath_40k.npz

In [ ]:
!python -m task1.train --config task1/configs/oplora_r16.yaml --out_root {RUNS} --data /content/data/metamath_40k.npz

In [ ]:
!python -m task1.train --config task1/configs/lora_r4.yaml --out_root {RUNS} --data /content/data/metamath_40k.npz

## Evaluation models

Merge adapters into fresh fp16 copies of the base model (projected factors for OPLoRA) and convert the full FT exports to fp16.
`merge_report.json` in each folder has the merged vs unmerged logit check.

In [ ]:
jobs = {
    'full_mid': ('full_ft', 625), 'full_final': ('full_ft', 1250), 'l2sp_final': ('full_ft_l2sp', 1250),
    'lora4_final': ('lora_r4', 1250), 'lora16_final': ('lora_r16', 1250), 'oplora16_final': ('oplora_r16', 1250),
}
for cond, (run, upd) in jobs.items():
    if not os.path.exists(f'{EVAL}/{cond}/config.json'):
        !python -m task1.merge --run_dir {RUNS}/{run} --update {upd} --out {EVAL}/{cond}

## Benchmarks (lm-eval)

Seven conditions. GSM8K generation is the slow part; each condition is skipped if its results already exist.

In [ ]:
import glob
conds = ['baseline', 'full_mid', 'full_final', 'l2sp_final', 'lora4_final', 'lora16_final', 'oplora16_final']
for cond in conds:
    model = 'Qwen/Qwen3-0.6B' if cond == 'baseline' else f'{EVAL}/{cond}'
    skip = [s for s in ['gsm8k', 'mc'] if glob.glob(f'{RES}/lm_eval/{cond}/{s}/**/results_*.json', recursive=True)]
    if len(skip) < 2:
        !python -m task1.evaluate --model {model} --name {cond} --out {RES}/lm_eval --skip {' '.join(skip)}

In [ ]:
!python -m task1.summarize_eval --lm_eval_dir {RES}/lm_eval --out {RES}

## Diagnostics, qualitative responses, plots

In [ ]:
!python -m task1.diagnostics --runs_dir {RUNS} --out {RES}/diagnostics
!python -m task1.qualitative --eval_dir {EVAL} --out {RES}/qualitative
!python -m task1.plot_curves --runs_dir {RUNS} --out {RES}/figures